# GFL Commercial — Route Profitability: Analysis (Part 2 evidence)

Reads the **Gold** Delta tables produced by the pipeline and answers the Part 2
questions. Sections 1–4 establish *what* "underperforming" means and *who* it is
(see `docs/spec.md` §1.3); sections 5–6 are the decision-relevant *why* and *trend*:

1. A flat margin threshold is indefensible — margins differ structurally by waste stream.
2. So underperformance is judged **within a cohort** (waste stream × customer segment).
3. Underperformance is **concentrated** (a few routes), not diffuse.
4. The losses are **structural, not episodic** (incidents / maintenance don't explain them).
5. The primary **cost driver** on low-margin route-days is **disposal cost**.
6. The **3-year trend** (2022–2024) — performance is gently improving, not deteriorating.

In [1]:
from lib import config
from pyspark.sql import functions as F

spark = config.get_spark("analysis-notebook")
spark.sparkContext.setLogLevel("ERROR")

fact_day  = spark.read.format("delta").load(str(config.FACT_ROUTE_DAY))
dim_route = spark.read.format("delta").load(str(config.DIM_ROUTE))
scorecard = spark.read.format("delta").load(str(config.ROUTE_SCORECARD))
df = fact_day.join(dim_route, "route_id")
print("route-days:", df.count(), "| routes:", dim_route.count())

26/06/07 13:13:16 WARN Utils: Your hostname, Anmols-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.68.50 instead (on interface en0)
26/06/07 13:13:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/satripleenkaur/.ivy2/cache
The jars for the packages stored in: /Users/satripleenkaur/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9c5ed99d-a9cf-4151-8244-7d5dea9b1132;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 53ms :: artifacts dl 2ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	----------------------------------------------------

:: loading settings :: url = jar:file:/Users/satripleenkaur/Repos/GFL-Assignment/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/06/07 13:13:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


route-days: 12000 | routes: 120


## 1. A flat margin threshold would be indefensible

Median margin by waste stream — the gap is about the **material**, not how the route is run.

In [2]:
(df.groupBy("primary_waste_stream")
   .agg(F.round(F.percentile_approx("gross_margin_pct", 0.5), 2).alias("median_margin_pct"),
        F.count(F.lit(1)).alias("route_days"))
   .orderBy("median_margin_pct")
   .toPandas())

,primary_waste_stream,median_margin_pct,route_days
0,General Waste,35.60,7357
1,Organics,42.77,967
2,Recycling,72.69,2866
3,Cardboard,76.52,810


General Waste sits ≈35.6% and Cardboard ≈76.5% — a ≈40-point spread. A single threshold
would condemn every General Waste route and excuse every Cardboard one.

## 2. So we benchmark within cohorts (waste stream × customer segment)

Each route is judged against the median of its own cohort.

In [3]:
cohort = (df.groupBy("cohort_key")
            .agg(F.round(F.percentile_approx("gross_margin_pct", 0.5), 2).alias("cohort_median_margin"),
                 F.count(F.lit(1)).alias("route_days")))
print("distinct cohorts:", cohort.count())
cohort.orderBy("cohort_median_margin").toPandas()

distinct cohorts: 21


,cohort_key,cohort_median_margin,route_days
0,General Waste | Office & Commercial,17.51,1556
1,Organics | Office & Commercial,19.02,200
2,General Waste | Retail,27.09,1900
3,General Waste | Hospitality,31.48,513
4,Organics | Retail,34.06,189
5,General Waste | Restaurant & Food Service,43.55,1196
6,Organics | Restaurant & Food Service,45.51,194
7,General Waste | Industrial,48.24,1645
8,General Waste | Healthcare,51.69,547
9,Organics | Industrial,52.07,189


## 3. Underperformance is concentrated, not diffuse

The `route_scorecard` verdict — most routes are fine; a small, actionable set is not.

In [4]:
scorecard.groupBy("tier").count().orderBy("tier").toPandas()

,tier,count
0,OK,98
1,Tier 1 - Loss-making,4
2,Tier 2 - Margin leak,18


In [5]:
(scorecard.filter("below_peer_flag")
   .select("route_id", "cohort_key",
           F.round("median_margin_pct", 1).alias("median_margin_pct"),
           F.round("cohort_median_margin", 1).alias("cohort_median_margin"),
           F.round("pct_days_below_peer", 2).alias("pct_days_below_peer"),
           F.round("loss_day_rate", 2).alias("loss_day_rate"),
           "tier")
   .orderBy(F.desc("loss_day_rate"), F.desc("pct_days_below_peer"))
   .toPandas())

,route_id,cohort_key,median_margin_pct,cohort_median_margin,pct_days_below_peer,loss_day_rate,tier
0,RTE-0035,General Waste | Office & Commercial,-10.6,17.5,0.96,0.78,Tier 1 - Loss-making
1,RTE-0114,General Waste | Retail,-14.8,27.1,0.99,0.75,Tier 1 - Loss-making
2,RTE-0061,General Waste | Retail,-5.5,27.1,0.98,0.58,Tier 1 - Loss-making
3,RTE-0059,General Waste | Office & Commercial,-2.1,17.5,0.80,0.52,Tier 1 - Loss-making
4,RTE-0117,General Waste | Office & Commercial,6.2,17.5,0.74,0.30,Tier 2 - Margin leak
5,RTE-0081,General Waste | Office & Commercial,6.8,17.5,0.73,0.30,Tier 2 - Margin leak
6,RTE-0070,General Waste | Retail,9.7,27.1,0.88,0.28,Tier 2 - Margin leak
7,RTE-0020,General Waste | Retail,10.4,27.1,0.86,0.27,Tier 2 - Margin leak
8,RTE-0069,General Waste | Retail,13.8,27.1,0.72,0.27,Tier 2 - Margin leak
9,RTE-0068,General Waste | Industrial,17.2,48.2,0.99,0.18,Tier 2 - Margin leak


22 of 120 routes are below their cohort on >70% of their days — underperformance is a
property of specific routes, which is what makes it actionable.

## 4. The losses are structural, not episodic

If losses were one-off events we'd expect them to coincide with incidents / maintenance spikes.

In [6]:
loss = df.filter(F.col("gross_profit") < 0)
n, nloss = df.count(), loss.count()
inc = loss.filter("incident_flag = 1").count()
print(f"loss-days:            {nloss} ({100*nloss/n:.1f}% of route-days)")
print(f"  with an incident:   {inc} ({100*inc/nloss:.1f}% of loss-days)")
print(f"  incident rate, all: {100*df.filter('incident_flag = 1').count()/n:.1f}%")

loss-days:            717 (6.0% of route-days)
  with an incident:   25 (3.5% of loss-days)
  incident rate, all: 2.5%


In [7]:
(df.withColumn("day_type", F.when(F.col("gross_profit") < 0, "loss day").otherwise("profitable day"))
   .groupBy("day_type")
   .agg(F.round(F.avg("maintenance_cost"), 2).alias("avg_maintenance_cost"),
        F.round(F.percentile_approx("maintenance_cost", 0.5), 2).alias("median_maintenance_cost"),
        F.count(F.lit(1)).alias("days"))
   .toPandas())

,day_type,avg_maintenance_cost,median_maintenance_cost,days
0,loss day,83.54,80.80,717
1,profitable day,78.68,76.32,11283


Only ≈3% of loss-days carry an incident, and maintenance cost on loss-days is in line with
normal days. The losses are baked into the route economics — so the fix is **re-pricing /
route redesign**, not chasing one-off events.

## 5. The primary cost driver on loss-days is disposal cost

Part 2 asks what *drives* low-margin route-days. Decomposing `total_cost` into its
components — and comparing loss-days against profitable days — points to one lever.

In [8]:
comp = ["disposal_cost", "fuel_cost", "labour_cost", "maintenance_cost", "admin_cost"]
g = df.withColumn("day_type", F.when(F.col("gross_profit") < 0, "loss day").otherwise("profitable day"))

# Each component as a share of total cost, loss-days vs profitable days.
shares = g.groupBy("day_type").agg(
    *[F.round(F.sum(c) * 100 / F.sum("total_cost"), 1).alias(c) for c in comp]
)
shares.toPandas()

,day_type,disposal_cost,fuel_cost,labour_cost,maintenance_cost,admin_cost
0,loss day,75.3,4.4,15.4,1.5,3.5
1,profitable day,65.2,5.5,19.1,1.9,8.3


In [9]:
# Absolute average per route-day — what actually differs on a loss-day.
g.groupBy("day_type").agg(
    F.round(F.avg("total_revenue"), 0).alias("avg_revenue"),
    *[F.round(F.avg(c), 0).alias(f"avg_{c}") for c in comp],
).toPandas()

,day_type,avg_revenue,avg_disposal_cost,avg_fuel_cost,avg_labour_cost,avg_maintenance_cost,avg_admin_cost
0,loss day,4879.0,4257.0,246.0,869.0,84.0,199.0
1,profitable day,8517.0,2696.0,228.0,791.0,79.0,342.0


**Disposal cost is the driver.** On loss-days it is **≈75%** of total cost vs **≈65%**
on profitable days, and the average disposal bill is **≈£4,257/day vs ≈£2,696** — yet
loss-days earn barely half the revenue (≈£4,879 vs ≈£8,517). Fuel, labour and
maintenance are essentially flat between the two. Low-margin route-days are a
**disposal-cost-vs-revenue** problem (heavy/low-value material, or under-priced tipping),
not a fleet-efficiency one — which is exactly why the Tier-1 fix is **re-pricing**, not routing.

## 6. Performance is gently improving across the 3-year window (2022–2024)

Part 2 asks whether things are getting better or worse. A revenue-weighted margin
(the correct way to aggregate a ratio) by year and quarter answers it.

In [10]:
dim_date = spark.read.format("delta").load(str(config.DIM_DATE))
dft = df.join(dim_date.select("date_key", "year", "quarter"), "date_key")

# Yearly: revenue-weighted margin + loss-day rate.
dft.groupBy("year").agg(
    F.round(F.sum("gross_profit") * 100 / F.sum("total_revenue"), 1).alias("margin_pct"),
    F.round(F.avg((F.col("gross_profit") < 0).cast("int")) * 100, 1).alias("loss_day_rate_pct"),
).orderBy("year").toPandas()

,year,margin_pct,loss_day_rate_pct
0,2022,48.3,6.8
1,2023,49.1,5.8
2,2024,49.8,5.4


In [11]:
# Quarterly trend — the same weighted margin across all 12 quarters.
(dft.groupBy("year", "quarter")
    .agg(F.round(F.sum("gross_profit") * 100 / F.sum("total_revenue"), 1).alias("margin_pct"))
    .orderBy("year", "quarter")
    .toPandas())

,year,quarter,margin_pct
0,2022,Q1,47.3
1,2022,Q2,48.9
2,2022,Q3,48.5
3,2022,Q4,48.3
4,2023,Q1,49.2
5,2023,Q2,49.7
6,2023,Q3,49.7
7,2023,Q4,47.7
8,2024,Q1,49.7
9,2024,Q2,49.6


**Improving, not deteriorating.** Fleet margin drifts up **48.3% → 49.1% → 49.8%**
across 2022–2024 and the loss-day rate falls **6.8% → 5.8% → 5.4%**; the quarterly series
is stable with a slight upward trend, ending Q4-2024 at ≈50.1%. The fleet as a whole is
healthy and slowly improving — which reinforces that the problem is **concentrated** in the
22 flagged routes, not a system-wide decline.

## Verdict
The Part 2 answer is `route_scorecard`: **4 Tier-1** routes (loss-making → re-price or cut)
and **18 Tier-2** routes (margin leak → optimise routing / sequencing / disposal).

In [12]:
spark.stop()